# Phần 2: Làm sạch & Feature Engineering

Dựa trên các phát hiện từ Phần 1 (EDA), notebook này thực hiện:
1. Loại bỏ dữ liệu lỗi (`price = 0`) và cột gây leakage (`price_per_sqft`)
2. Tạo feature mới: tuổi nhà, đã renovate hay chưa, tổng số phòng...
3. Encode `city` (one-hot, đã gộp nhóm hiếm), loại `statezip` vì trùng thông tin với `city`

> Toàn bộ logic thực tế nằm trong `src/preprocessing.py` để có thể tái sử dụng
> ở `train.py`. Notebook này chỉ gọi lại và trực quan hóa từng bước.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

from data_loader import load_raw_data
from preprocessing import clean_data, engineer_features, encode_categorical

df_raw = load_raw_data('../data/raw/modified_data.csv')
print('Shape ban đầu:', df_raw.shape)

## Bước 1: Làm sạch dữ liệu

- Bỏ các dòng `price = 0` (49 dòng, lỗi nhập liệu — không phải nhà miễn phí)
- Bỏ cột `price_per_sqft`: đây là **data leakage** vì nó được tính trực tiếp
  bằng `price / sqft_living`. Nếu giữ lại, model sẽ "gian lận" bằng cách học
  ngược công thức này thay vì học từ đặc điểm thực sự của căn nhà.
- Bỏ cột `street`: quá chi tiết (gần như mỗi căn nhà một địa chỉ riêng),
  không tổng quát hoá được, one-hot sẽ tạo hàng nghìn cột vô nghĩa.

In [ ]:
df_clean = clean_data(df_raw)
print('Shape sau khi làm sạch:', df_clean.shape)
print('Số dòng đã loại bỏ:', df_raw.shape[0] - df_clean.shape[0])
df_clean.head()

## Bước 2: Feature Engineering

Các feature mới được tạo và lý do:

| Feature | Công thức | Lý do |
|---|---|---|
| `house_age` | year_sold - yr_built | Tuổi nhà thường ảnh hưởng giá hơn là năm xây tuyệt đối |
| `was_renovated` | 1 nếu yr_renovated > 0 | yr_renovated=0 nghĩa là 'chưa renovate', cần tách thành cờ nhị phân rõ ràng |
| `years_since_renovation` | năm bán - năm renovate (hoặc house_age nếu chưa renovate) | Nhà mới renovate gần đây có thể bán giá cao hơn |
| `total_rooms` | bedrooms + bathrooms | Một chỉ số tổng gộp đơn giản, đôi khi robust hơn từng biến riêng |
| `year_sold`, `month_sold` | tách từ `date` | Giá nhà có thể biến động theo mùa/năm bán |

In [ ]:
df_feat = engineer_features(df_clean)
df_feat[['house_age', 'was_renovated', 'years_since_renovation', 'total_rooms',
         'year_sold', 'month_sold']].describe()

## Bước 3: Encode biến phân loại

**Vấn đề phát hiện khi thử nghiệm ban đầu:** one-hot cả `city` VÀ `statezip`
cùng lúc tạo ra 138 cột, khiến Linear/Ridge Regression **overfit nặng**
(R² âm trên tập test) vì:
- `statezip` gần như là thông tin trùng lặp với `city` (mỗi mã zip chỉ
  thuộc về đúng 1 thành phố) → đa cộng tuyến.
- Nhiều thành phố/zip chỉ có vài mẫu → cột one-hot gần như toàn 0, dễ khiến
  model học nhiễu (noise) thay vì tín hiệu thật.

**Cách khắc phục** (xem `encode_categorical()` trong `src/preprocessing.py`):
1. Bỏ hẳn `statezip`, chỉ giữ `city`.
2. Gộp các thành phố có ít hơn 30 mẫu thành nhóm `'Other'` trước khi one-hot.

Kết quả: số chiều giảm từ 138 xuống 43, và quan trọng hơn — Linear/Ridge
Regression từ R² âm đã lên **R²(log) ≈ 0.68**, ngang bằng Random Forest.

In [ ]:
print('Số thành phố trước khi gộp:', df_feat['city'].nunique())

df_final = encode_categorical(df_feat)
print('Shape sau encode:', df_final.shape)

city_cols = [c for c in df_final.columns if c.startswith('city_')]
print('Số cột city sau one-hot (đã gộp nhóm hiếm):', len(city_cols))

## Bước 4: Lưu dữ liệu đã xử lý

Trong thực tế chỉ cần chạy `python src/preprocessing.py` từ terminal — hàm
`run_pipeline()` đã gộp cả 3 bước trên và lưu kết quả vào
`data/processed/cleaned_data.csv`.

In [ ]:
from preprocessing import run_pipeline

df_final = run_pipeline(save=True)
df_final.head()